# Opptra Pricing Signal POC — ML + Agentic

**The brief (Ranjit, Category Ops):** *"I need something that shows me the problem and suggests a fix — not something that just shows me more data."*

This notebook is the **streamlined product build** behind the Angular UI in `frontend/`. It turns a snapshot of 8 SKUs into **decisions**, not a dashboard.

Pipeline (each step = one section below):

1. **Ingest** the 8-SKU snapshot into a structured frame.
2. **ML model** — a tiny logistic-regression *Buy Box win-probability* model. It learns how price position maps to winning the Buy Box, so we can ask *"at what price do we likely win?"* instead of guessing.
3. **Triage** — classify every SKU into `LOST_RECOVERABLE`, `WON_HEADROOM`, `BELOW_FLOOR`, `HOLD` (mirrors `pricing.service.ts`).
4. **Agentic layer (Azure AI Foundry)** — an LLM agent writes a specific, actionable recommendation, **hard-constrained by the margin floor** with a guardrail loop that rejects any sub-floor suggestion.
5. **Apply** — one-click action simulation that flips the SKU to `Repriced`.

**Edge case SKU-007:** competitor (₹399) is *below* our floor (₹420) → flagged `BELOW_FLOOR`, no action, never priced below floor.

> Runtime: `Runtime → Run all`. Add Foundry secrets in **Section 4** (or it falls back to a deterministic template so the notebook still runs end-to-end without keys).

## 0. Setup

In [ ]:
%pip install -q pandas scikit-learn openai azure-ai-inference

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.9/124.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 15.5 MB/s eta 0:00:00


## 1. Ingest & parse the pricing data

Hardcoded snapshot — same 8 SKUs the UI ships with (`frontend/src/app/data/skus.ts`). A working prototype with hardcoded data beats a broken uploader.

In [1]:
import pandas as pd

SKU_SNAPSHOT = [
    {"id": "SKU-001", "brand": "Natura Casa",   "ourPrice": 1299, "competitorPrice": 1199, "buyBox": "Lost", "marginFloor": 1050},
    {"id": "SKU-002", "brand": "Natura Casa",   "ourPrice":  849, "competitorPrice":  860, "buyBox": "Won",  "marginFloor":  720},
    {"id": "SKU-003", "brand": "LivSpace Pro",  "ourPrice": 2499, "competitorPrice": 2199, "buyBox": "Lost", "marginFloor": 1800},
    {"id": "SKU-004", "brand": "LivSpace Pro",  "ourPrice":  599, "competitorPrice":  610, "buyBox": "Won",  "marginFloor":  480},
    {"id": "SKU-005", "brand": "Artisan Home",  "ourPrice": 3799, "competitorPrice": 3750, "buyBox": "Lost", "marginFloor": 3200},
    {"id": "SKU-006", "brand": "Artisan Home",  "ourPrice": 1150, "competitorPrice": 1390, "buyBox": "Won",  "marginFloor":  900},
    {"id": "SKU-007", "brand": "Nordic Basics", "ourPrice":  449, "competitorPrice":  399, "buyBox": "Lost", "marginFloor":  420},
    {"id": "SKU-008", "brand": "Nordic Basics", "ourPrice": 2199, "competitorPrice": 2100, "buyBox": "Lost", "marginFloor": 1750},
]

df = pd.DataFrame(SKU_SNAPSHOT)
df["priceGap"] = df["competitorPrice"] - df["ourPrice"]      # negative = competitor undercut us
df["marginHeadroom"] = df["ourPrice"] - df["marginFloor"]    # room to drop while staying profitable
df

,id,brand,ourPrice,competitorPrice,buyBox,marginFloor,priceGap,marginHeadroom
0,SKU-001,Natura Casa,1299,1199,Lost,1050,-100,249
1,SKU-002,Natura Casa,849,860,Won,720,11,129
2,SKU-003,LivSpace Pro,2499,2199,Lost,1800,-300,699
3,SKU-004,LivSpace Pro,599,610,Won,480,11,119
4,SKU-005,Artisan Home,3799,3750,Lost,3200,-49,599
5,SKU-006,Artisan Home,1150,1390,Won,900,240,250
6,SKU-007,Nordic Basics,449,399,Lost,420,-50,29
7,SKU-008,Nordic Basics,2199,2100,Lost,1750,-99,449


## 2. ML model — Buy Box win probability

We fit a small **logistic regression**: feature = price position vs the competitor (`(competitor − ours) / competitor`), label = did we win the Buy Box. With marketplaces the Buy Box is largely a function of relative price, so even this tiny model captures the core dynamic and gives us a **continuous win-probability curve** per SKU.

Why ML and not a hard rule? It lets the agent answer *"what is the **highest** price that still wins?"* (recover Buy Box with minimum margin loss) and *"how much can I raise before I risk losing it?"* — both are search problems over a learned probability, not a single if-statement.

In [2]:
import numpy as np
from sklearn.linear_model import LogisticRegression

def price_position(our_price, competitor_price):
    """Relative undercut as a percentage: >0 means we are cheaper than the competitor.
    Expressed in percent so the feature has enough magnitude for the model to learn a
    sharp win/lose boundary (raw fractions are ~0.1 and get washed out by regularisation)."""
    return (competitor_price - our_price) / competitor_price * 100.0

X = df.apply(lambda r: price_position(r.ourPrice, r.competitorPrice), axis=1).to_numpy().reshape(-1, 1)
y = (df["buyBox"] == "Won").astype(int).to_numpy()

# Tiny dataset -> add light regularisation so the curve is smooth, not a step function.
model = LogisticRegression(C=2.0).fit(X, y)

def win_probability(our_price, competitor_price):
    feat = np.array([[price_position(our_price, competitor_price)]])
    return float(model.predict_proba(feat)[0, 1])

df["winProbNow"] = df.apply(lambda r: round(win_probability(r.ourPrice, r.competitorPrice), 2), axis=1)
df[["id", "ourPrice", "competitorPrice", "buyBox", "winProbNow"]]


,id,ourPrice,competitorPrice,buyBox,winProbNow
0,SKU-001,1299,1199,Lost,0.00
1,SKU-002,849,860,Won,0.86
2,SKU-003,2499,2199,Lost,0.00
3,SKU-004,599,610,Won,0.92
4,SKU-005,3799,3750,Lost,0.21
5,SKU-006,1150,1390,Won,1.00
6,SKU-007,449,399,Lost,0.00
7,SKU-008,2199,2100,Lost,0.00


In [3]:
WIN_TARGET = 0.55  # probability we consider "likely to win the Buy Box"

def best_recover_price(competitor_price, margin_floor, win_target=WIN_TARGET):
    """Highest price >= floor that the model expects to win the Buy Box.
    Returns None if no price at/above floor clears the win target (true BELOW_FLOOR case).
    """
    candidates = range(int(np.ceil(margin_floor)), int(competitor_price) + 1)
    winners = [p for p in candidates if win_probability(p, competitor_price) >= win_target]
    return max(winners) if winners else None

def safe_raise_price(competitor_price, current_price, margin_floor, win_target=WIN_TARGET):
    """Highest price we can raise to while staying above floor and still likely winning."""
    ceiling = int(competitor_price)
    candidates = range(int(current_price), ceiling + 1)
    winners = [p for p in candidates if p >= margin_floor and win_probability(p, competitor_price) >= win_target]
    return max(winners) if winners else None

## 3. Triage — surface the signal, not the noise

Same four categories as the production service. The ML model decides whether a Lost item is *actually* recoverable above floor.

In [4]:
HEADROOM_THRESHOLD = 50  # min gap below competitor before a Won item is worth raising

def triage(sku):
    comp, ours, floor, bb = sku.competitorPrice, sku.ourPrice, sku.marginFloor, sku.buyBox
    competitor_below_floor = comp < floor

    if competitor_below_floor:
        return "BELOW_FLOOR", None
    if bb == "Lost":
        target = best_recover_price(comp, floor)
        return ("LOST_RECOVERABLE", target) if target else ("BELOW_FLOOR", None)
    # Won the Buy Box: is there meaningful room to raise price?
    if comp - ours >= HEADROOM_THRESHOLD:
        return "WON_HEADROOM", safe_raise_price(comp, ours, floor)
    return "HOLD", None

cats, suggested = zip(*df.apply(triage, axis=1))
df["category"] = cats
df["suggestedPrice"] = suggested
df["competitorBelowFloor"] = df["competitorPrice"] < df["marginFloor"]

order = {"LOST_RECOVERABLE": 0, "WON_HEADROOM": 1, "BELOW_FLOOR": 2, "HOLD": 3}
triage_view = df.sort_values(by="category", key=lambda c: c.map(order))
triage_view[["id", "brand", "ourPrice", "competitorPrice", "marginFloor", "category", "suggestedPrice"]]

,id,brand,ourPrice,competitorPrice,marginFloor,category,suggestedPrice
0,SKU-001,Natura Casa,1299,1199,1050,LOST_RECOVERABLE,1199.0
2,SKU-003,LivSpace Pro,2499,2199,1800,LOST_RECOVERABLE,2199.0
4,SKU-005,Artisan Home,3799,3750,3200,LOST_RECOVERABLE,3750.0
7,SKU-008,Nordic Basics,2199,2100,1750,LOST_RECOVERABLE,2100.0
5,SKU-006,Artisan Home,1150,1390,900,WON_HEADROOM,1390.0
6,SKU-007,Nordic Basics,449,399,420,BELOW_FLOOR,NaN
1,SKU-002,Natura Casa,849,860,720,HOLD,NaN
3,SKU-004,LivSpace Pro,599,610,480,HOLD,NaN


In [5]:
actionable = df[df["category"].isin(["LOST_RECOVERABLE", "WON_HEADROOM"])]
blocked = df[df["category"] == "BELOW_FLOOR"]
print(f"Actionable now: {len(actionable)}   |   Blocked (below floor): {len(blocked)}   |   Holding: {len(df[df['category']=='HOLD'])}")
print("Actionable SKUs:", ", ".join(actionable["id"]))
print("Blocked SKUs   :", ", ".join(blocked["id"]), "<- SKU-007 lands here (competitor below floor)")

Actionable now: 5   |   Blocked (below floor): 1   |   Holding: 2
Actionable SKUs: SKU-001, SKU-003, SKU-005, SKU-006, SKU-008
Blocked SKUs   : SKU-007 <- SKU-007 lands here (competitor below floor)


## 4. Agentic recommendation layer — Azure AI Foundry

The agent receives the **triage signal + the ML-derived suggested price** and writes a one-sentence, manager-ready recommendation with the tradeoff.

Two guardrails make this *real* AI work, not filler:

1. **Constrained input** — the model is told the margin floor and the ML target price explicitly.
2. **Validation loop** — we parse the agent's price back out and **reject anything below floor**, retrying with a stricter instruction. A recommendation below floor is a failure, so it can never ship one.

Set your Foundry endpoint/key below. In Colab: `🔑 Secrets` → add `FOUNDRY_ENDPOINT`, `FOUNDRY_API_KEY`, `FOUNDRY_MODEL`.

In [6]:
import os
from pathlib import Path

# --- Azure AI Foundry config -------------------------------------------------
# Foundry exposes an OpenAI-compatible endpoint, e.g.
#   https://<your-resource>.services.ai.azure.com/openai/v1
#
# Secrets are loaded from (in priority order):
#   1. Colab secrets (FOUNDRY_ENDPOINT / FOUNDRY_API_KEY / FOUNDRY_MODEL)
#   2. environment variables
#   3. a local `.env` file next to this notebook (git-ignored; never commit keys)
#
# Create scripts/.env with:
#   FOUNDRY_ENDPOINT=https://<your-resource>.services.ai.azure.com/openai/v1
#   FOUNDRY_API_KEY=<your-key>
#   FOUNDRY_MODEL=gpt-4.1


def _load_dotenv():
    for candidate in (Path.cwd() / ".env", Path.cwd() / "scripts" / ".env"):
        if candidate.exists():
            for line in candidate.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
            break


_load_dotenv()

try:
    from google.colab import userdata  # type: ignore
    _get = lambda k: userdata.get(k) or os.environ.get(k)
except Exception:
    _get = lambda k: os.environ.get(k)

FOUNDRY_ENDPOINT = _get("FOUNDRY_ENDPOINT")
FOUNDRY_API_KEY = _get("FOUNDRY_API_KEY")
FOUNDRY_MODEL = _get("FOUNDRY_MODEL") or "gpt-4.1"

USE_FOUNDRY = bool(FOUNDRY_ENDPOINT and FOUNDRY_API_KEY)
print("Foundry configured:", USE_FOUNDRY, "| model:", FOUNDRY_MODEL)
if not USE_FOUNDRY:
    print("No Foundry credentials found -> using deterministic template fallback.")


Foundry configured: True | model: gpt-4.1


In [7]:
import re

SYSTEM_PROMPT = (
    "You are a pricing co-pilot for an e-commerce marketplace team. "
    "You write ONE short, decisive recommendation a pricing manager can act on immediately. "
    "HARD RULE: never recommend a price below the margin floor. "
    "Always state the exact price, why, and the tradeoff (margin or Buy Box). "
    "Be specific with numbers. No hedging, no 'consider'. Under 40 words."
)

def _build_user_prompt(sku):
    return (
        f"SKU {sku.id} ({sku.brand}). Our price Rs.{sku.ourPrice}, competitor Rs.{sku.competitorPrice}, "
        f"margin floor Rs.{sku.marginFloor}, Buy Box {sku.buyBox}. "
        f"Triage: {sku.category}. ML-recommended price: Rs.{int(sku.suggestedPrice)}. "
        f"Win probability at that price is high. Write the recommendation. "
        f"The price you state MUST equal Rs.{int(sku.suggestedPrice)} and MUST be >= Rs.{sku.marginFloor}."
    )

def _foundry_chat(messages):
    from openai import OpenAI
    client = OpenAI(base_url=FOUNDRY_ENDPOINT, api_key=FOUNDRY_API_KEY)
    resp = client.chat.completions.create(model=FOUNDRY_MODEL, messages=messages, temperature=0.2, max_tokens=120)
    return resp.choices[0].message.content.strip()

def _template_reco(sku):
    """Deterministic fallback so the notebook runs without Foundry keys."""
    price = int(sku.suggestedPrice)
    margin = price - sku.marginFloor
    if sku.category == "LOST_RECOVERABLE":
        return (f"Set {sku.id} to Rs.{price} — Rs.{sku.competitorPrice - price} below competitor, "
                f"Rs.{margin} above floor. Recovers the Buy Box while protecting margin.")
    return (f"Raise {sku.id} to Rs.{price} — still Rs.{sku.competitorPrice - price} under competitor and "
            f"Rs.{margin} above floor. Captures headroom without risking the Buy Box.")

def _extract_price(text):
    nums = re.findall(r"Rs\.?\s*([0-9][0-9,]*)", text)
    return int(nums[0].replace(",", "")) if nums else None

def recommend(sku, max_retries=2):
    """Agentic loop: generate -> validate against margin floor -> retry if it breaches."""
    if sku.suggestedPrice is None:
        return None
    if not USE_FOUNDRY:
        return _template_reco(sku)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": _build_user_prompt(sku)},
    ]
    for attempt in range(max_retries + 1):
        text = _foundry_chat(messages)
        price = _extract_price(text)
        if price is not None and price >= sku.marginFloor:
            return text  # passed the guardrail
        # Guardrail tripped: push back and retry.
        messages.append({"role": "assistant", "content": text})
        messages.append({"role": "user", "content": (
            f"That price violates the margin floor of Rs.{sku.marginFloor}. "
            f"Rewrite using exactly Rs.{int(sku.suggestedPrice)}.")})
    return _template_reco(sku)  # all retries failed -> safe deterministic output

In [8]:
# Generate recommendations for every actionable SKU (urgency: Lost first).
actionable_sorted = actionable.sort_values(
    by=["category", "priceGap"], key=lambda c: c.map(order) if c.name == "category" else c
)

recommendations = {}
for sku in actionable_sorted.itertuples():
    text = recommend(sku)
    recommendations[sku.id] = text
    print(f"[{sku.category}] {text}\n")

[LOST_RECOVERABLE] Set price to Rs.2199 to match the competitor and recover the Buy Box; this maintains a margin of Rs.399 above the floor, trading some margin for a high probability of Buy Box win.

[LOST_RECOVERABLE] Set price to Rs.1199 to match the competitor and recover the Buy Box; this maintains a margin of Rs.149 above the floor. Tradeoff: margin decreases, but Buy Box win probability is high.

[LOST_RECOVERABLE] Set price to Rs.2100 to match the competitor and recover the Buy Box; margin remains above floor at Rs.350. Tradeoff: margin drops by Rs.99, but Buy Box win probability is high. Act now.

[LOST_RECOVERABLE] Set price to Rs.3750 to match the competitor and recover the Buy Box. This maintains a margin of Rs.550 per unit, trading some margin for higher sales volume and Buy Box win probability.

[WON_HEADROOM] Raise price to Rs.1390 to maximize margin while maintaining Buy Box leadership, as win probability remains high and margin floor of Rs.900 is protected. Tradeoff: hi

In [9]:
# Edge-case proof: SKU-007 gets a flag, never a price.
for sku in blocked.itertuples():
    print(f"[BELOW_FLOOR] {sku.id}: competitor Rs.{sku.competitorPrice} is below floor Rs.{sku.marginFloor}. "
          f"No profitable action — matching the competitor would lose money. Flag for manual review.")

[BELOW_FLOOR] SKU-007: competitor Rs.399 is below floor Rs.420. No profitable action — matching the competitor would lose money. Flag for manual review.


## 5. One-click action simulation

`apply_reprice()` mirrors the UI's **Apply** button: it flips state to `Repriced`, records the new price, and confirms — no real marketplace API, but the interaction is action-oriented. The floor is re-checked one last time before any write.

In [10]:
df["status"] = "Pending"

def apply_reprice(sku_id):
    row = df.loc[df["id"] == sku_id].iloc[0]
    new_price = row["suggestedPrice"]
    if new_price is None:
        return f"❌ {sku_id}: no action available (below floor)."
    if new_price < row["marginFloor"]:
        return f"🚫 {sku_id}: blocked — Rs.{int(new_price)} is below floor Rs.{row['marginFloor']}."
    df.loc[df["id"] == sku_id, ["ourPrice", "status"]] = [int(new_price), "Repriced"]
    return f"✅ {sku_id} repriced to Rs.{int(new_price)} — live on marketplace (simulated)."

# Apply every actionable recommendation.
for sku_id in actionable_sorted["id"]:
    print(apply_reprice(sku_id))

✅ SKU-003 repriced to Rs.2199 — live on marketplace (simulated).
✅ SKU-001 repriced to Rs.1199 — live on marketplace (simulated).
✅ SKU-008 repriced to Rs.2100 — live on marketplace (simulated).
✅ SKU-005 repriced to Rs.3750 — live on marketplace (simulated).
✅ SKU-006 repriced to Rs.1390 — live on marketplace (simulated).


In [11]:
df[["id", "brand", "ourPrice", "competitorPrice", "marginFloor", "category", "suggestedPrice", "status"]]

,id,brand,ourPrice,competitorPrice,marginFloor,category,suggestedPrice,status
0,SKU-001,Natura Casa,1199,1199,1050,LOST_RECOVERABLE,1199.0,Repriced
1,SKU-002,Natura Casa,849,860,720,HOLD,NaN,Pending
2,SKU-003,LivSpace Pro,2199,2199,1800,LOST_RECOVERABLE,2199.0,Repriced
3,SKU-004,LivSpace Pro,599,610,480,HOLD,NaN,Pending
4,SKU-005,Artisan Home,3750,3750,3200,LOST_RECOVERABLE,3750.0,Repriced
5,SKU-006,Artisan Home,1390,1390,900,WON_HEADROOM,1390.0,Repriced
6,SKU-007,Nordic Basics,449,399,420,BELOW_FLOOR,NaN,Pending
7,SKU-008,Nordic Basics,2100,2100,1750,LOST_RECOVERABLE,2100.0,Repriced


## Summary

| Step | What it does | Where it lives in the product |
|---|---|---|
| Ingest | 8-SKU snapshot → structured frame | `data/skus.ts` |
| ML | Logistic win-probability → best price above floor | new layer this notebook proves |
| Triage | 4-way classification, urgency-sorted | `pricing.service.ts` |
| Agent | Foundry LLM, floor-constrained + retry guardrail | replaces rule-based `suggestPrice` |
| Apply | State flip to `Repriced` | `price-bar` component |

**SKU-007** is handled explicitly: competitor below floor → `BELOW_FLOOR`, flagged, never priced. **Next 4 hours:** stream live competitor prices, persist the win model on real Buy Box history, and wire `recommend()` behind the Angular service via a thin API.